In [52]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [53]:
api_key=os.getenv('PINECONE_API_KEY')

In [54]:
from langchain_community.retrievers import PineconeHybridSearchRetriever


In [55]:
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=api_key)

if "my-index" not in pc.list_indexes().names():
    pc.create_index(
    name="my-index",
    dimension=384,
    metric="dotproduct",
    spec=ServerlessSpec(
        cloud="aws",
        region="us-east-1"
    )
)

index = pc.Index("my-index")

In [56]:
# index=pc.index(index_name)
# index

In [57]:
from langchain_huggingface import HuggingFaceEmbeddings
hf_api_key=os.getenv('HF_TOKEN')
embeddins=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
embeddins

HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [58]:
from pinecone_text.sparse import BM25Encoder

bm25_encoder = BM25Encoder()

sentences = [
    "In 2023, I visited Paris",
    "In 2022, I visited New York",
    "In 2021, I visited New Orleans",
]

bm25_encoder.fit(sentences)

  0%|          | 0/3 [00:00<?, ?it/s]

In [59]:
retriever=PineconeHybridSearchRetriever(embeddings=embeddins,sparse_encoder=bm25_encoder,index=index)


In [60]:
retriever

PineconeHybridSearchRetriever(embeddings=HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False), sparse_encoder=<pinecone_text.sparse.bm25_encoder.BM25Encoder object at 0x000001AC0263F450>, index=<pinecone.data.index.Index object at 0x000001AC027DF510>)

In [61]:
vec = embeddins.embed_query("hello")
print(len(vec))

384


In [62]:
retriever.add_texts(
    [
    "In 2023, I visited Paris",
        "In 2022, I visited New York",
        "In 2021, I visited New Orleans",

]
)

  0%|          | 0/1 [00:00<?, ?it/s]

In [63]:
print(pc.describe_index("my-index"))

{'deletion_protection': 'disabled',
 'dimension': 384,
 'host': 'my-index-w6dt3tq.svc.aped-4627-b74a.pinecone.io',
 'metric': 'dotproduct',
 'name': 'my-index',
 'spec': {'serverless': {'cloud': 'aws', 'region': 'us-east-1'}},
 'status': {'ready': True, 'state': 'Ready'},
 'tags': None,
 'vector_type': 'dense'}


In [64]:
# pc.delete_index("my-index")

In [65]:
retriever.invoke("What city did i visit first")


[Document(metadata={'score': 0.232818261}, page_content='In 2022, I visited New York'),
 Document(metadata={'score': 0.21249935}, page_content='In 2023, I visited Paris'),
 Document(metadata={'score': 0.239368573}, page_content='In 2021, I visited New Orleans')]